# Reproducing Kalugin et al. (2026) from dandiset 001074

Paper. Kalugin PN, et al. (2026). Simultaneous, real-time tracking of many neuromodulatory
signals with multiplexed optical recording of sensors on a micro-endoscope. *Neuron*.
[doi:10.1016/j.neuron.2026.07.008](https://doi.org/10.1016/j.neuron.2026.07.008). The claims file
was written from the accepted author manuscript in PubMed Central (PMC13568791) with its
supplement (Figures S1 to S6); the version of record was not read.

Dandiset. [DANDI:001074](https://dandiarchive.org/dandiset/001074/0.260928.0807), version
0.260928.0807. The paper cites the dandiset without a version. Three versions are published
(0.260804.2042, 0.260807.0137, 0.260928.0807); their asset lists hold the same 713 paths with the
same blobs and sizes (checked through the API while preparing), so they do not differ in anything
the analysis uses. The most recent is pinned, as the protocol asks when the paper cites none.

What the dandiset holds. For each MORSE recording, the CaImAn output of the paper's pipeline:
a trace per cell mask (the denoised trace and its residual) and the 3D masks. There are no sensor
labels, no well, wash-on, injection or stimulation times, no region or experiment labels, and no
deformation fields, raw imaging, photometry, ELISA, histology or choroid plexus traces. Subjects
are named by date and probe, with the context (`wells`, `slice`, `mouse`).

Outside input. The well order of the ten revision probes' identification and dose-response
dipping, the mixture concentrations, and the baseline and normalization wells of each fit come
from the paper's code, github.com/pkalugin/kalugin-et-al-morse at commit
fa4526ac02eaa2c5a24c4cfdcaab5c36ed5ae4ed (`6_trace_analysis/morse_figs3_rev.ipynb`). They name
the frame blocks of recordings the dandiset holds and bring in no measurements, so the claims
that use them are `with_labels` and depend on the link claim `link-revision-well-order`.

| Claim | Tested here |
|---|---|
| link-revision-well-order (link, added by the author) | tested |
| fig1c-cells-per-gel | tested (curated masks only) |
| fig1g-sensor-yield, fig1g-all-ten-sensors-respond, fig1h-crosstalk-pairs | tested |
| table1-ec50-on-target, table1-ec50-off-target, table1-peak-dff-morse | tested |
| fig1j-ec50-consistent-across-probes, figs3e-tau-on-under-one-minute, figs3f-depth-delay | tested |
| figs2f-cells-tracked | tested |
| fig1d, fig1f, methods-revision-probe-stability, figs2c, slices-probe-stability | blocked, data missing (deformation fields, raw imaging) |
| figs6ef, fig4-ex-vivo-elevations, fig4-ex-vivo-two-orders-lower, figs6j, fig5gh, fig6, fig6c, figs6a | blocked, data missing (photometry, ELISA and mass spectrometry, choroid plexus, histology) |
| the 16 slice claims (fig2c to fig2l, figs4de, figs5c to figs5i, slices-no-peptide-release) | blocked, metadata insufficient |
| fig3df, fig3g, fig3h, fig4-ip-csf-concentrations, fig5ab, fig5cd | blocked, metadata insufficient |

In [ ]:
import time
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import morse as M

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
started = time.time()
print("dandiset", M.DANDISET, "version", M.VERSION)

## Data check

The survey reads the shapes and metadata of all 713 files, and one value of each trace table to
find files deposited twice.

In [ ]:
t0 = time.time()
S = pd.DataFrame(M.survey())
S["kind"] = S.path.map(M.kind_of)
S["context"] = S.subject.str.extract(r"(wells|slice|mouse)")[0].fillna("none")
print(f"{len(S)} files read in {time.time() - t0:.0f} s; {S.subject.nunique()} subjects")
print("\nFiles by context and kind:")
print(pd.crosstab(S.kind, S.context).to_string())
print("\nMetadata values over all files:")
for c in ["rate_hz", "dt", "t0", "indicator", "start", "accepted", "rejected", "acquisition", "stimulus"]:
    print(f"  {c}: {sorted(S[c].unique().tolist())[:6]}")
print("\nFrames per file: min", S.frames.min(), "max", S.frames.max(),
      "; cells per file: min", S.cells.min(), "max", S.cells.max())
print("Mask volume planes (z) per file:", sorted(Counter(s[2] for s in S.mask_shape).items()))

Every file states an imaging rate of 30 Hz, timestamps from 0 in steps of 1/30 s, the indicator
"OGB-1", and the same session start (2023-01-01 12:30). These are placeholders: the Methods give
about 0.5 volumes per second, the paper's code uses 31 volumes per minute (`mean_factor` 31), and
the sensors are GRAB sensors. The frame index is the only clock the files have; at 31 frames per
minute the recordings run from 1.2 to 314 minutes. No mask is marked accepted or rejected.
No file has acquisition or stimulus data. The mask volumes have 16 to 26 planes, where the Methods
describe 30 planes per volume.

In [ ]:
S["frames_mod_93"] = S.frames % M.FRAMES_PER_WELL
w = S[S.context == "wells"]
print("Dipping (wells) files whose frame count is a multiple of 93 (one well = 3 min):",
      int((w.frames_mod_93 == 0).sum()), "of", len(w))
print("Frame counts of the dipping files that are not:", sorted(w.frames[w.frames_mod_93 != 0].unique().tolist()))
dup = S.groupby(["frames", "cells", "first_values"]).path.apply(list)
dup = dup[dup.str.len() > 1]
print(f"\nGroups of files with identical shape and first trace values: {len(dup)}")
for v in dup:
    print("  ", [p.split("/")[-1].replace("_ophys.nwb", "").replace("ses-ch0-means-movie-nobg-cnmf-", "") for p in v])
print("\nThe 67 files of subject '0' (no name):", int((S.subject == "0").sum()),
      "; of them, duplicates of a named file:",
      sum(any(p.startswith("sub-0/") for p in v) and any(not p.startswith("sub-0/") for p in v) for v in dup))

254 of the 384 dipping files have frame counts that are multiples of 93, consistent with 3-minute
wells at 31 frames per minute; the other 130 (34%), listed above, are not. All twenty recordings
of the ten revision probes are. Files deposited twice are listed above: the unnamed subject `0`
repeats named files, and the slice and wells001 files of probe 250918-SL1ZS5 are identical, so one
of the two recordings of that probe is missing and the other is deposited under both names. Each
recording is counted once below. The masks are stored scrambled; the cell below shows it on one
mask.

In [ ]:
p = M.file_path("251003-PL65", "wells001")
with M.open_nwb(p) as f:
    m = f["processing/ophys/ImageSegmentation/PlaneSegmentation/image_mask"][0]
nz = np.argwhere(m > 0)
print("mask 0 of", p.split("/")[-1], "shape", m.shape)
print("  as stored: nonzero voxels", len(nz), "bounding box", nz.min(0).tolist(), "to", nz.max(0).tolist())
v = m.ravel().reshape(m.shape, order="F")
nz = np.argwhere(v > 0)
print("  reordered (Fortran order): bounding box", nz.min(0).tolist(), "to", nz.max(0).tolist())
print("  per-plane weight as stored:", np.round(m.sum(axis=(0, 1)), 2).tolist())
print("  per-plane weight reordered:", np.round(v.sum(axis=(0, 1)), 2).tolist())

Read as stored, the mask covers the full width of the volume with the same weight in every plane.
Reordered as a Fortran-order volume it is a compact blob of about 25 by 30 pixels over about 12
planes, as a cell should be. The masks were written from CaImAn's column-major vectors without
the reordering; every depth below uses the reordered masks.

## The revision probes and the well order (link-revision-well-order)

> The code's well lists for the ten revision probes (identification: AVP, 5-HT, NE, ACh, ATP, OXT,
> HA, DA, CCK, MT, PBS; dose response: the A5NAA mixture at seven 10-fold steps, the OHDCM mixture
> at seven steps, PBS) apply to the dandiset's wells001 and wells101 recordings of those probes.

The matched files of the identification (wells001) and dose (wells101) recordings are used, as in
morse_figs3_rev cells 5 and 7. Traces are C plus the residual (the code's `CY_matched_raw`).

In [ ]:
t0 = time.time()
R = M.load_rev()
rows = []
for p in M.REV_PROBES:
    rows.append({"probe": p, "id_cells": R[p]["id"].shape[0], "id_frames": R[p]["id"].shape[1],
                 "id_wells": R[p]["id"].shape[1] / 93, "dose_cells": R[p]["dose"].shape[0],
                 "dose_frames": R[p]["dose"].shape[1], "dose_wells": R[p]["dose"].shape[1] / 93})
rev = pd.DataFrame(rows)
print(f"loaded in {time.time() - t0:.0f} s")
print(rev.to_string(index=False))
print("cells over the ten probes:", rev.id_cells.sum())

In [ ]:
pre = [M.id_preprocess(R[p]["id"]) for p in M.REV_PROBES]
PN = np.vstack([x[1] for x in pre])
act = M.derivative(PN).argmax(axis=1)
within = act % 93
print("cells:", len(act), "; exponential background fitted for", int(sum(x[2].sum() for x in pre)),
      "(the rest activate within the first two wells)")
print("Frame of each cell's steepest rise within its well (0 = onset), counts by bin:")
print(pd.Series(pd.cut(within, [-1, 10, 20, 30, 46, 62, 92])).value_counts(sort=False).to_string())
frac = float(np.mean(within < 31))
print(f"Fraction rising in the first minute of a well: {frac:.2f} (one third if rises fell anywhere in a well)")
for shift in [0, 15, 31, 46]:
    fr = float(np.mean(((act - shift) % 93) < 31))
    print(f"  with well onsets shifted by {shift:2d} frames: {fr:.2f}")

Sensor labels (SENSOR IDENTITY of the claims file, the dataset giving none). The Methods'
procedure: dF/F0 (Eq. 27), the jump artefact removed, an exponential background fitted before
activation, peak normalization (Eq. 28), a Savitzky-Golay derivative over 1 minute (31 frames;
the claims file's header gives 30 frames, shown as a variant), HDBSCAN (min_cluster_size 3) on
correlation distance, and clusters matched to ligands by correlation
with the time course of ligand administration. Reading (a) pools the cells of all ten probes in
one clustering; reading (b) adds, for cells (a) leaves unlabelled, the label of their cluster in a
clustering of their own probe, as the code's second pass does (morse_figs3_rev cells 62-69). The
code then corrects clusters by hand (cells 73-78), which cannot be repeated.

Two details of the cluster matching are assumptions, since the Methods do not give them: the time
course of ligand administration is a boxcar over the first minute of each well, and a cluster keeps
its ligand only when its mean derivative also peaks in that ligand's well (the peak rule, the
author's own, meant to drop weak matches). Both are shown changed below: without the peak rule, and
with a boxcar over the whole well. Reading (b) is used in the sections that follow, and every
section also shows its result under reading (a).

In [ ]:
la, lb, grp, der = M.identify([x[1] for x in pre])
LAB = lb
READINGS = [("b", lb), ("a", la)]
print("labelled cells, reading (a):", int((la != "").sum()), " reading (b):", int((lb != "").sum()), "of", len(lb))
tab = pd.DataFrame({"a": pd.Series(la[la != ""]).value_counts(), "b": pd.Series(lb[lb != ""]).value_counts()}).reindex(M.SENSORS)
print(tab.to_string())
# The two assumptions of the cluster matching: the peak rule and the boxcar width.
la_np, lb_np, _, _ = M.identify([x[1] for x in pre], require_peak=False)
la_box, lb_box, _, _ = M.identify([x[1] for x in pre], width=M.FRAMES_PER_WELL)
print("\nlabelled cells under the variants of the cluster matching:")
print(pd.DataFrame([{"variant": "main (1-min boxcar, peak rule)", "(a)": int((la != "").sum()), "(b)": int((lb != "").sum())},
                    {"variant": "without the peak rule", "(a)": int((la_np != "").sum()), "(b)": int((lb_np != "").sum())},
                    {"variant": "boxcar over the whole well", "(a)": int((la_box != "").sum()), "(b)": int((lb_box != "").sum())}]).to_string(index=False))
print("cells the peak rule leaves unlabelled: (a)", int(((la == "") & (la_np != "")).sum()),
      "; (b)", int(((lb == "") & (lb_np != "")).sum()),
      "; cells whose label the rule changes (b):", int(((lb != "") & (lb_np != "") & (lb != lb_np)).sum()))

In [ ]:
DX = np.vstack([M.remove_jumps(M.dff(R[p]["dose"])) for p in M.REV_PROBES])
PLm = M.well_plateaus(DX)
rows = []
for s in M.SENSORS:
    r = LAB == s
    a = PLm[r][:, 0:7].max(axis=1) - PLm[r][:, 0]
    o = PLm[r][:, 7:14].max(axis=1) - PLm[r][:, 7]
    own = "A" if s in M.MIX_A else "O"
    hit = (a > o) if own == "A" else (o > a)
    rows.append({"sensor": s, "mixture": own, "cells": int(r.sum()), "larger rise in own mixture": int(hit.sum()),
                 "fraction": round(float(hit.mean()), 2) if r.sum() else np.nan})
link = pd.DataFrame(rows)
print("Dose recording: rise of each cell's well plateaus (mean of the third minute) over the first well of each mixture block")
print(link.to_string(index=False))
nx = link[~link.sensor.isin(["AVP", "OXT", "NE", "DA"])]
print("Fraction over sensors without crosstalk:", round(nx["larger rise in own mixture"].sum() / nx.cells.sum(), 2))
print("Fraction over all labelled cells:", round(link["larger rise in own mixture"].sum() / link.cells.sum(), 2))

In [ ]:
# Label check: mean peak-normalized identification trace of each sensor's cells, averaged over each well,
# and the rise of its plateau (mean of the third minute) over the previous well's plateau (the first well
# over the mean of its first 10 frames). A label is consistent with its ligand when the largest rise is in
# the ligand's well.
WELL_MEAN, WELL_INC = {}, {}
for lname, L in READINGS:
    wm, wi = {}, {}
    for s in M.SENSORS:
        m = PN[L == s].mean(axis=0)
        wm[s] = [m[w * 93:(w + 1) * 93].mean() for w in range(11)]
        pl = np.array([m[w * 93 + 62:(w + 1) * 93].mean() for w in range(11)])
        wi[s] = np.r_[pl[0] - m[:10].mean(), np.diff(pl)]
    WELL_MEAN[lname] = pd.DataFrame(wm, index=M.ID_WELLS).T
    WELL_INC[lname] = pd.DataFrame(wi, index=M.ID_WELLS).T
    print(f"reading ({lname}): mean peak-normalized trace per well")
    print(WELL_MEAN[lname].round(2).to_string())
    print(f"reading ({lname}): rise of the well plateau over the previous well's")
    print(WELL_INC[lname].round(2).to_string())
    big = WELL_INC[lname].idxmax(axis=1)
    print(f"reading ({lname}): sensors whose largest rise is in their own ligand's well:",
          int(sum(big[s] == s for s in M.SENSORS)), "of 10;", {s: big[s] for s in M.SENSORS if big[s] != s})
    print()
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, (lname, _) in zip(axs, READINGS):
    im = ax.imshow(WELL_MEAN[lname].values, vmin=0, vmax=1, cmap="viridis", aspect="auto")
    ax.set_xticks(range(11), M.ID_WELLS, rotation=45)
    ax.set_yticks(range(10), M.SENSORS)
    ax.set_title(f"reading ({lname}): mean per well")
fig.colorbar(im, ax=axs, label="peak-normalized dF/F0")
plt.show()
print("plotted range:", round(min(WELL_MEAN[k].values.min() for k in WELL_MEAN), 2), "to",
      round(max(WELL_MEAN[k].values.max() for k in WELL_MEAN), 2), "(colour scale 0 to 1)")

**Result: reproduced.** All ten identification recordings have 1023 frames (11 wells of 93) and
all ten dose recordings 1395 frames (15 wells), as the code's well lists require. 86% of cells rise
most steeply in the first minute of a well, against 23%, 11% and 3% with the well onsets shifted by
15, 31 and 46 frames. In the dose recording, 79% of the labelled cells (77% for the sensors without
crosstalk, and 73% to 92% for each sensor except DA) rise more in the mixture block that holds
their ligand; the DA sensor cells, which also respond to the NE of the other block, do so in 41%.
The label check agrees: under both labelling readings, the plateau of every sensor's mean trace
rises most in the well of its own ligand (10 of 10), by 0.47 to 0.74 of the peak-normalized range
in reading (b). This check is not independent of the labels, which are made from the time of each
cell's steepest rise. The peak rule leaves 47 cells unlabelled in reading (b) (18 in reading a), and
changes 3 labels.

## fig1c-cells-per-gel

> Imaging a MORSE hydrogel through the GRIN lens shows 100 to 200 green sensor cells spread
> through the full thickness of the gel.

Reading (a) of the criterion, the CNMF components before curation, is not in the dandiset: its
files hold the curated masks (`mr-raw`, merged and refit), the masks after a further manual merge
(`mr-m-raw`) and the matched masks. Reading (b) is tested on the `mr-raw` files, one per recording,
and shown also on `mr-m-raw`. Depth uses the reordered masks of the `mr-raw` identification
recordings of the five revision probes that have one, and, as a variant covering all ten probes, the
matched masks of their dose recordings; the gel thickness is read as the imaged z range,
an assumption (the claims file does not say how the thickness is bounded).

In [ ]:
U = S[(S.subject != "0")].copy()
U = U[~((U.subject == "250918-SL1ZS5-wells001"))]  # deposited identical to its slice recording
U["probe"] = U.subject.str.replace(r"-(wells\d*|slice|mouse\d?)$", "", regex=True)
for k in ["mr-raw", "mr-m-raw"]:
    d = U[U.kind == k]
    per_probe = d.groupby("probe").cells.mean()
    inr = ((per_probe >= 90) & (per_probe <= 220))
    print(f"{k}: {len(d)} recordings of {d.probe.nunique()} probes; masks per recording median {d.cells.median():.0f}, "
          f"range {d.cells.min()} to {d.cells.max()}")
    print(f"   probes whose mean count lies in 90-220: {int(inr.sum())} of {len(inr)} ({inr.mean():.2f})")
    print("   by context:", d.groupby("context").cells.agg(["count", "median", "min", "max"]).to_dict("index"))

In [ ]:
t0 = time.time()
rows = []
for p in M.REV_PROBES:
    path = M.file_path(p, "wells001", "mr-raw")
    if path not in M.assets():
        continue
    wz = M.mask_z_profiles(path)
    z = M.z_centre(wz)
    nz = wz.shape[1]
    third = np.floor(z / (nz / 3)).clip(0, 2).astype(int)
    c = np.bincount(third, minlength=3)
    rows.append({"recording": p + " wells001 mr-raw", "masks": len(z), "planes": nz, "z min": round(z.min(), 1),
                 "z max": round(z.max(), 1), "top third": c[0], "middle third": c[1], "bottom third": c[2]})
# Variant: the matched masks of the dose recording, which all ten probes have (used again in figs3f).
ZD, NZ = {}, {}
for p in M.REV_PROBES:
    wz = M.mask_z_profiles(M.file_path(p, "wells101"))
    ZD[p], NZ[p] = M.z_centre(wz), wz.shape[1]
    third = np.floor(ZD[p] / (NZ[p] / 3)).clip(0, 2).astype(int)
    c = np.bincount(third, minlength=3)
    rows.append({"recording": p + " wells101 matched", "masks": len(ZD[p]), "planes": NZ[p], "z min": round(ZD[p].min(), 1),
                 "z max": round(ZD[p].max(), 1), "top third": c[0], "middle third": c[1], "bottom third": c[2]})
depth1c = pd.DataFrame(rows)
print(f"masks read in {time.time() - t0:.0f} s")
print(depth1c.to_string(index=False))
allthirds = (depth1c[["top third", "middle third", "bottom third"]] > 0).all(axis=1)
print("recordings with cells in all three thirds: curated masks", int(allthirds[depth1c.recording.str.endswith("mr-raw")].sum()),
      "of", int(depth1c.recording.str.endswith("mr-raw").sum()), "; dose-recording matched masks",
      int(allthirds[depth1c.recording.str.endswith("matched")].sum()), "of", int(depth1c.recording.str.endswith("matched").sum()))

**Result: reproduced, under reading (b) and the assumption that the imaged z range is the gel
thickness.** Over 59 recordings of 43 probes the curated masks number 21 to 205 per recording
(median 128), and 38 of 43 probes (0.88) have a mean count within 90 to 220; with the further
merged masks (`mr-m-raw`), 31 of 38. In each of the five identification recordings read, mask
centres fall in the top (5 to 21), middle (44 to 88) and bottom (27 to 67) thirds of the imaged
volume, and so do the matched masks of the dose recordings of all ten revision probes (top 3 to
21, middle 18 to 79, bottom 3 to 66). These volumes have 17 to 24 planes, against the 30 planes of
the Methods, so the thirds are of the imaged range, not necessarily of the gel. Reading (a), the
CNMF components before curation, cannot be tested: they are not in the dandiset.

## fig1g-sensor-yield

> Ten ten-sensor probes yielded 506 identified single-cell masks; per probe a median of 9 sensors
> were active (IQR 8 to 9), with on average 6.02 cells per sensor per probe.

A sensor is active on a probe when at least one cell carries its label; cells per sensor per probe
is the number of labelled cells over the number of (probe, sensor) pairs with a cell
(morse_figs3_rev cells 139-144). The code prints sensors per probe 5, 8, 9, 10, 6, 9, 10, 9, 9, 9
(cell 140), shown for comparison. The derivative window of the code (30 frames) is shown as a
variant of the Methods' 1 minute (31 frames), and so are the two assumptions of the cluster
matching (without the peak rule; a boxcar over the whole well).

In [ ]:
code_counts = [5, 8, 9, 10, 6, 9, 10, 9, 9, 9]
rows = []
la30, lb30, _, _ = M.identify([x[1] for x in pre], window=30)
for name, lab in [("(a) pooled", la), ("(b) pooled + per probe", lb), ("(a) window 30", la30), ("(b) window 30", lb30),
                  ("(a) without the peak rule", la_np), ("(b) without the peak rule", lb_np),
                  ("(a) whole-well boxcar", la_box), ("(b) whole-well boxcar", lb_box)]:
    y = M.yield_stats(lab, grp)
    rows.append({"reading": name, **{k: (round(v, 2) if isinstance(v, float) else v) for k, v in y.items() if k != "sensors_per_probe"},
                 "sensors per probe": y["sensors_per_probe"]})
print(pd.DataFrame(rows).to_string(index=False))
print("code's sensors per probe:", code_counts)
print("Expected: 506 cells (455-557), median 9 (8.1-9.9), 25th percentile 8 (7.2-8.8), 75th 9 (8.1-9.9), 6.02 (5.42-6.62)")
print("\nCells per sensor and probe, reading (b):")
print(pd.crosstab(pd.Series(lb[lb != ""], name="sensor"), pd.Series(np.array(M.REV_PROBES)[grp[lb != ""]], name="probe")).reindex(M.SENSORS).fillna(0).astype(int).to_string())

**Result: partially reproduced, resting on the labelling.** The sensors per probe match: median 9
(reading b; 8.5 in reading a), 25th percentile 8 and 75th 9, and the per-probe counts
[5, 8, 9, 10, 5, 9, 10, 9, 8, 9] differ from the code's [5, 8, 9, 10, 6, 9, 10, 9, 9, 9] by one
sensor on two probes. The cell counts do not: 559 labelled cells in reading (b), just above the 455
to 557 range, and 384 in reading (a); 6.82 and 4.74 cells per sensor per probe against 5.42 to
6.62. No variant brings both into range: the code's 30-frame window gives 550 and 383 cells (6.71
and 4.73), dropping the peak rule 606 and 402 (7.21 and 4.90), and a whole-well boxcar 539 and 372
(6.57 and 4.59). The code's hand corrections, which drop and move cells, are not repeated.

## fig1g-all-ten-sensors-respond

> Dipping ten-sensor probes into each ligand in turn gave rapid, strong and consistent responses
> in a subset of cells for each ligand, and all ten sensors were activated.

For each sensor, the mean over its cells of the peak-normalized trace; in which well is it highest
(the maximum within each well, and as a variant the mean over each well)? The criterion allows the
DA/NE and AVP/OXT partners. Because the labels are made from the time of each cell's steepest rise,
this test is not independent of the labelling; it shows where the mean signal peaks. Since
responses do not wash out between wells, a variant asks in which well the plateau of the mean trace
rises most over the previous well's (the increments of the label check above). Both labelling
readings are shown.

In [ ]:
partner = {"AVP": "OXT", "OXT": "AVP", "NE": "DA", "DA": "NE"}
for lname, L in READINGS:
    rows = []
    for s in M.SENSORS:
        r = L == s
        m = PN[r].mean(axis=0)
        wmax = [m[w * 93:(w + 1) * 93].max() for w in range(11)]
        wmean = [m[w * 93:(w + 1) * 93].mean() for w in range(11)]
        own = M.ID_WELLS.index(s)
        ok = {s, partner.get(s, s)}
        inc = WELL_INC[lname].loc[s]
        rows.append({"sensor": s, "cells": int(r.sum()), "probes": len(set(grp[r])),
                     "own-well max": round(wmax[own], 2), "highest well (max)": M.ID_WELLS[int(np.argmax(wmax))],
                     "its value": round(max(wmax), 2), "highest well (mean)": M.ID_WELLS[int(np.argmax(wmean))],
                     "own or partner (max)": M.ID_WELLS[int(np.argmax(wmax))] in ok,
                     "own or partner (mean)": M.ID_WELLS[int(np.argmax(wmean))] in ok,
                     "own, partner or next well (max)": M.ID_WELLS[int(np.argmax(wmax))] in ok | {M.ID_WELLS[own + 1]},
                     "largest rise": inc.idxmax(), "own-well rise": round(inc[s], 2), "largest rise in own or partner": inc.idxmax() in ok})
    resp = pd.DataFrame(rows)
    print(f"reading ({lname})")
    print(resp.to_string(index=False))
    print("  sensors with cells on at least one probe:", int((resp.cells > 0).sum()), "of 10")
    print("  sensors whose mean peaks in their own (or partner's) well: by the per-well maximum",
          int(resp["own or partner (max)"].sum()), "of 10; by the per-well mean", int(resp["own or partner (mean)"].sum()), "of 10")
    print("  allowing the well after the ligand's (the code's choice for AVP, NE and CCK, cell 248):",
          int(resp["own, partner or next well (max)"].sum()), "of 10")
    print("  variant, the largest rise of the well plateau over the previous well's in the own or partner's well:",
          int(resp["largest rise in own or partner"].sum()), "of 10\n")

**Result: not reproduced.** All ten sensors carry labels, each on 5 to 10 of the ten probes. As the
criterion is written, the mean peak-normalized trace of a sensor's cells is highest in its own or
its partner's well for 2 of 10 sensors (OXT, ATP) by the per-well maximum, and 3 of 10 by the
per-well mean (adding ACh), under both labelling readings; allowing the well after the ligand's, as
the code does for AVP, NE and CCK, 7 of 10. For CCK and MT the maximum is in the PBS well, for NE,
DA and ACh in the ATP well. The responses persist into later wells: the AVP sensor's mean is 0.71
in the AVP well and 0.88 in the next. Measured as the rise of each well's plateau over the previous
well's, every sensor rises most in its own ligand's well (10 of 10, under both readings). The
failure is of the criterion's "highest during the well", which carry-over defeats; whether that is
the reading of Figure 1G intended is asked of a reader (questions/001074.yaml).

## fig1h-crosstalk-pairs

> Among the ten sensors, only the DA and NE sensors, and to a lesser extent the AVP and OXT
> sensors, show crosstalk in their ligand responses.

Per cell, the derivative of the peak-normalized trace (Savitzky-Golay, 1 minute, order 1; the
code's 10 frames as variant), divided by its maximum; the mean per sensor; the Pearson correlation
at zero lag between the ten means. The band-pass artefact variant of cell 219 is not run.
Both labelling readings are shown.

In [ ]:
for lname, L in READINGS:
    for win in [31, 10]:
        d = np.vstack([M.derivative(x[1], win) for x in pre])
        d = d / d.max(axis=1, keepdims=True)
        means = np.array([d[L == s].mean(axis=0) for s in M.SENSORS])
        C = pd.DataFrame(np.corrcoef(means), index=M.SENSORS, columns=M.SENSORS)
        iu = np.triu_indices(10, 1)
        pairs = sorted([(C.values[i, j], M.SENSORS[i], M.SENSORS[j]) for i, j in zip(*iu)], reverse=True)
        print(f"reading ({lname}), window {win} frames: largest off-diagonal correlations")
        for v, a, b in pairs[:5]:
            print(f"   {a}-{b}: {v:.3f}")
        rank = [k for k, (v, a, b) in enumerate(pairs) if {a, b} == {"AVP", "OXT"}][0] + 1
        above = [f"{a}-{b}" for v, a, b in pairs[:rank - 1] if {a, b} != {"DA", "NE"}]
        print(f"   AVP-OXT: {C.loc['AVP', 'OXT']:.3f}, rank {rank} of 45; pairs other than DA-NE above it: {len(above)}")
        if win == 31 and lname == "b":
            print(C.round(2).to_string())

**Result: not reproduced.** The DA-NE correlation of the sensors' mean derivatives is the
largest off-diagonal value in every variant (0.77 with the 1-minute window and 0.83 with the code's
10 frames in reading b; 0.63 and 0.70 in reading a). The AVP-OXT correlation is not the second: it
is 0.16, rank 12 of 45 (0.43, rank 14, with 10 frames) in reading (b), and rank 15 and 16 in
reading (a). Ten to fourteen pairs other than DA-NE lie above it, led by 5-HT-DA (0.46) and DA-ACh
(0.45), so neither the AVP-OXT part nor the "only" of the claim holds.

## table1-ec50-on-target and table1-ec50-off-target

> The MORSE EC50 of each sensor for its own ligand, from consensus dose-response fits, is AVP
> 355 nM, OXT 2.19 uM, 5-HT 687 nM, CCK 245 nM, NE 4.04 uM, DA 1.70 uM, ACh 7.03 uM, ATP 75.2 uM,
> MT 22.0 nM and HA 6.39 uM.

> The crosstalk sensors respond to the partner ligand with EC50s of 28.7 uM (AVP sensor to OXT),
> 247 nM (OXT sensor to AVP), 9.35 uM (NE sensor to DA) and 8.20 uM (DA sensor to NE).

Dose recording of each labelled cell: dF/F0, jump artefact removed. Plateau per well: the mean of
the third minute (Methods), with the code's median of sorted frames 47-77 as variant. The
normalization is open in the Methods ("corrected with an exponentially decaying background fit and
then peak-normalized"); three readings are run: (i) the code's, the baseline-well plateau
subtracted and divided by the peak-well plateau, from the baseline well on (morse_figs3_rev cells
338-339, 365); (ii) the same with the code's plateau; (iii) Eq. 28 over the dose recording after
an exponential background fitted before activation, all seven wells. Eq. 9 with Hill coefficient
1 is fitted by least squares to the cells of all probes pooled. The code's noise cut and its
pooling with an older probe set for ACh, ATP, MT and HA: the noise cut is run as variant (iv)
(reading i on the cells it keeps, cells 343 and 365); the older probe set is not part of this
claim's ten probes and is not run. The code's per-cell background fits with hand-set windows
(cells 285-293) cannot be repeated. Eq. 9 has no offset, while the Eq. 28 plateaus of reading (iii)
do not start at zero; its fits fall on the lower bound of 1 pM for some groups, so a variant (iii-b)
subtracts each cell's plateau in the lowest well before fitting. Both labelling readings are shown.

In [ ]:
EXP_EC50 = {("AVP", "AVP"): 355, ("OXT", "OXT"): 2190, ("5HT", "5HT"): 687, ("CCK", "CCK"): 245, ("NE", "NE"): 4040,
            ("DA", "DA"): 1700, ("ACh", "ACh"): 7030, ("ATP", "ATP"): 75200, ("MT", "MT"): 22.0, ("HA", "HA"): 6390,
            ("AVP", "OXT"): 28700, ("OXT", "AVP"): 247, ("NE", "DA"): 9350, ("DA", "NE"): 8200}
PLc = M.well_plateaus(DX, "sorted_median")
LOWER_BOUND_NM = 1e-12 * 1e9 * 1.5  # fits within 1.5x of the fit's lower bound (1 pM) sit on the bound


def ec50_table(L):
    rows, fits = [], {}
    for g in M.GROUP_ORDER:
        r = L == g[0]
        out = {"sensor": g[0], "ligand": g[1], "cells": int(r.sum()), "paper nM": EXP_EC50[g]}
        for name, PL in [("i", PLm), ("ii", PLc)]:
            c, y = M.group_points(PL[r], g, "code")
            kd, f, n = M.fit_group(c, y)
            fits[(name, g)] = (kd, f)
            out[f"({name}) nM"] = round(kd * 1e9, 1)
            out[f"({name}) F"] = round(f, 2)
        keep = M.noise_keep(M.normalized_dose_trace(DX[r], g))
        c, y = M.group_points(PLm[r][keep], g, "code")
        kd, f, n = M.fit_group(c, y)
        fits[("iv", g)] = (kd, f)
        out["(iv) cells"], out["(iv) nM"], out["(iv) F"] = int(keep.sum()), round(kd * 1e9, 1), round(f, 2)
        x = DX[r]
        b = M.block_of(g[1]) * 93
        a = b + M.derivative(M.peak_norm(x))[:, b:b + 651].argmax(axis=1)
        xb, _ = M.exp_background(x, a)
        c, y = M.group_points(M.well_plateaus(M.peak_norm(xb)), g, "eq28")
        kd, f, n = M.fit_group(c, y)
        out["(iii) nM"], out["(iii) F"] = round(kd * 1e9, 3), round(f, 2)
        # Eq. 9 has no offset; variant (iii-b) subtracts each cell's plateau in the lowest well first
        kd, f, n = M.fit_group(c, y - y[:, [0]])
        out["(iii-b) nM"], out["(iii-b) F"] = round(kd * 1e9, 1), round(f, 2)
        for name in ["i", "ii", "iii", "iii-b", "iv"]:
            out[f"({name}) within 10%"] = abs(out[f"({name}) nM"] - EXP_EC50[g]) <= 0.1 * EXP_EC50[g]
        rows.append(out)
    return pd.DataFrame(rows), fits


ECS, FITS_BY = {}, {}
for lname, L in READINGS:
    EC, fits = ec50_table(L)
    ECS[lname], FITS_BY[lname] = EC, fits
    print(f"labels, reading ({lname})")
    print(EC.to_string(index=False))
    on = EC[EC.sensor == EC.ligand]
    offt = EC[EC.sensor != EC.ligand]
    for name in ["i", "ii", "iii", "iii-b", "iv"]:
        print(f"  normalization ({name}): on-target within 10%: {int(on[f'({name}) within 10%'].sum())} of 10; "
              f"off-target: {int(offt[f'({name}) within 10%'].sum())} of 4; "
              f"median |log10 ratio| on-target {np.median(np.abs(np.log10(on[f'({name}) nM'] / on['paper nM']))):.2f}")
    print("  normalization (iii): fits on the lower bound of 1 pM:", int((EC["(iii) nM"] < LOWER_BOUND_NM).sum()), "of 14\n")
EC, FITS = ECS["b"], FITS_BY["b"]

In [ ]:
print("Median normalized plateau per well (reading i), by group; concentrations in log10 M")
for g in M.GROUP_ORDER:
    r = LAB == g[0]
    c, y = M.group_points(PLm[r], g, "code")
    print(f"  {g[0]:>3} by {g[1]:<3} log10 c {np.round(np.log10(c), 0).astype(int).tolist()}  median y {np.round(np.nanmedian(y, axis=0), 2).tolist()}  "
          f"range of y {np.nanmin(y):.1f} to {np.nanmax(y):.1f}")

**Result: not reproduced.** With the reading (b) labels, the code's normalization and the
Methods' plateau (reading i), 2 of 10 EC50s are within 10% of Table 1 (OXT 2217 nM, CCK 266 nM);
the others differ by up to a factor of 75 (5-HT 9.2 nM against 687 nM). With the code's plateau
(ii), 0 of 10; with Eq. 28 over the recording (iii), 0 of 10, three of its fits on the 1 pM bound;
with the lowest-well plateau subtracted (iii-b), 0 of 10; with the code's noise cut (iv), 2 of 10
(5-HT 632 nM, DA 1622 nM), and the median distance from Table 1 falls from 0.37 to 0.12 log10 units.
With the reading (a) labels, 2, 1, 0, 1 and 3 of 10 (median distance 0.11 to 0.67 log10 units). The
median normalized plateaus rise from 0 to about 1 over the wells of every group, while single cells
reach -35 to 161, and the pooled least-squares fits follow those cells.

**Result: not reproduced.** None of the four off-target EC50s is within 10% under any
normalization or either labelling reading. Normalizations (i) and (iv), reading (b) labels: AVP
sensor to OXT 1339 and 558 nM against 28.7 uM; OXT sensor to AVP 32.5 and 20.2 nM against 247 nM;
NE sensor to DA 562 and 2957 nM against 9.35 uM; DA sensor to NE 20.1 and 4.38 uM against 8.20 uM.
The off-target groups are normalized to their own peak well; since F is free in the fit, this
does not change the EC50.

## table1-peak-dff-morse

> On MORSE probes the peak dF/F0 of each sensor to a saturating dose of its ligand is 0.37 (AVP),
> 0.24 (OXT), 0.36 (5-HT), 0.40 (CCK), 0.34 (NE), 0.27 (DA), 0.37 (ACh), 0.28 (ATP), 0.30 (MT) and
> 0.38 (HA), and to the partner ligand 0.10 (AVP sensor, OXT), 0.05 (OXT sensor, AVP), 0.15 (NE
> sensor, DA) and 0.21 (DA sensor, NE).

Identification recording, dF/F0 after the jump and background corrections; a cell's peak is the
maximum within the third minute of the well (morse_figs3_rev cell 160); the value is the median
over cells with a positive peak (cells 248, 263). Wells: (a) the ligand's well; (b) the code's
well (cell 248: the well after the ligand's for AVP, NE and CCK; for the OXT sensor's response to
AVP, the well after the AVP well). Both labelling readings are shown.

In [ ]:
X = np.vstack([x[0] for x in pre])
EXP_PK = {("AVP", "AVP"): 0.37, ("AVP", "OXT"): 0.10, ("OXT", "OXT"): 0.24, ("OXT", "AVP"): 0.05, ("5HT", "5HT"): 0.36,
          ("CCK", "CCK"): 0.40, ("NE", "NE"): 0.34, ("NE", "DA"): 0.15, ("DA", "DA"): 0.27, ("DA", "NE"): 0.21,
          ("ACh", "ACh"): 0.37, ("ATP", "ATP"): 0.28, ("MT", "MT"): 0.30, ("HA", "HA"): 0.38}
W = {s: M.ID_WELLS.index(s) for s in M.ID_WELLS}
code_well = {("AVP", "AVP"): 1, ("NE", "NE"): 3, ("CCK", "CCK"): 9, ("OXT", "AVP"): 1}
for lname, L in READINGS:
    rows = []
    for g, e in EXP_PK.items():
        r = L == g[0]
        out = {"sensor": g[0], "ligand": g[1], "paper": e}
        for name, w in [("a", W[g[1]]), ("b", code_well.get(g, W[g[1]]))]:
            v = X[r][:, w * 93 + 62:w * 93 + 93].max(axis=1)
            pos = v[v > 0]
            out[f"({name}) well"] = M.ID_WELLS[w]
            out[f"({name}) median"] = round(float(np.median(pos)), 3)
            out[f"({name}) cells>0"] = f"{len(pos)}/{len(v)}"
            out[f"({name}) within 10%"] = abs(np.median(pos) - e) <= 0.1 * e + 1e-9
        rows.append(out)
    pk = pd.DataFrame(rows)
    if lname == "b":
        PK = pk
    print(f"labels, reading ({lname}); wells (a) the ligand's, (b) the code's")
    print(pk.to_string(index=False))
    for name in "ab":
        print(f"  well reading ({name}): {int(pk[f'({name}) within 10%'].sum())} of 14 within 10%")
    print()

**Result: not reproduced.** With the reading (b) labels, 4 of 14 medians are within 10% of
Table 1 in the ligand's well (AVP 0.363, 5-HT 0.384, ACh 0.397, HA 0.400) and 5 of 14 in the code's
wells (adding NE 0.338). Of the others, all but NE (0.303) and the OXT sensor to AVP (0.044) are
higher than Table 1: OXT 0.292 against 0.24, CCK 0.492 against 0.40, DA 0.314 against 0.27, ATP
0.338 against 0.28, MT 0.564 against 0.30, and the crosstalk values 0.114, 0.207 and 0.236 against
0.10, 0.15 and 0.21. With the reading (a) labels, 2 of 14 in either well, and every on-target
median is higher than Table 1 (up to MT 0.827).

## fig1j-ec50-consistent-across-probes

> Dose-response relationships, after peak normalization and averaging over cells of the same
> sensor, are precise and consistent across probes.

Per-probe EC50s are fitted as in table1-ec50-on-target, reading (i), to each probe's cells alone,
for probes with more than one cell of the sensor (morse_figs3_rev cell 662). Readings (a) every
per-probe EC50 lies within a tenth to ten times the consensus EC50; (b) the SEM across probes is
less than half the mean. Since the consensus fits here can be degenerate (5-HT 9.2 nM), reading (a)
is also shown against the Table 1 EC50 (a'). Both labelling readings are shown.

In [ ]:
for lname, L in READINGS:
    fits = FITS_BY[lname]
    rows = []
    for g in M.GROUP_ORDER:
        if g[0] != g[1]:
            continue
        cons = fits[("i", g)][0]
        paper = M.KD_TABLE1[g]
        kds, kn = [], []
        for p in range(10):
            r = (L == g[0]) & (grp == p)
            if r.sum() > 1:
                c, y = M.group_points(PLm[r], g, "code")
                kds.append(M.fit_group(c, y)[0])
                keep = M.noise_keep(M.normalized_dose_trace(DX[r], g))
                if keep.sum() > 1:
                    c, y = M.group_points(PLm[r][keep], g, "code")
                    kn.append(M.fit_group(c, y)[0])
        kds, kn = np.array(kds), np.array(kn)
        consn = fits[("iv", g)][0]
        mean, sem = kds.mean(), kds.std(ddof=1) / np.sqrt(len(kds))
        rows.append({"sensor": g[0], "probes": len(kds), "consensus nM": round(cons * 1e9, 1),
                     "per-probe nM": np.round(kds * 1e9, 0).tolist(), "mean nM": round(mean * 1e9, 0), "SEM nM": round(sem * 1e9, 0),
                     "(a) all within 10x": bool(np.all((kds > cons / 10) & (kds < cons * 10))),
                     "(a) probes within": int(np.sum((kds > cons / 10) & (kds < cons * 10))),
                     "(a') all within 10x of Table 1": bool(np.all((kds > paper / 10) & (kds < paper * 10))),
                     "(b) SEM < mean/2": bool(sem < mean / 2),
                     "noise cut: probes": len(kn),
                     "noise cut (a)": bool(np.all((kn > consn / 10) & (kn < consn * 10))),
                     "noise cut (b)": bool(kn.std(ddof=1) / np.sqrt(len(kn)) < kn.mean() / 2) if len(kn) > 1 else False})
    j = pd.DataFrame(rows)
    if lname == "b":
        J = j
    print(f"labels, reading ({lname})")
    print(j.to_string(index=False))
    print("  reading (a) holds for", int(j["(a) all within 10x"].sum()), "of 10 sensors; (a') against Table 1 for",
          int(j["(a') all within 10x of Table 1"].sum()), "of 10; reading (b) for", int(j["(b) SEM < mean/2"].sum()), "of 10")
    print("  with the noise cut: (a)", int(j["noise cut (a)"].sum()), "of 10; (b)", int(j["noise cut (b)"].sum()), "of 10\n")

**Result: not reproduced.** With the reading (b) labels, every per-probe EC50 lies within a tenth
to ten times the consensus for 2 of 10 sensors (ACh, HA; reading a), or within that range of Table
1 for 3 of 10 (a'), and the SEM is below half the mean for 6 of 10 (reading b). With the code's
noise cut, 7 and 8 of 10. With the reading (a) labels, 6 (a), 6 (a') and 8 (b) of 10. Neither
reading holds for all sensors in any variant, so the criterion (both must hold) is not met, and
the outcome does not hinge on the two readings differing. Per-probe values span orders of magnitude
for some sensors (DA from 69 nM to 1.05 mM).

## figs3e-tau-on-under-one-minute

> Ligands diffuse quickly through the hydrogel; almost all sensors show an onset time constant
> under 1 minute, and most signals plateau within 1 minute.

Per cell, the dose trace normalized as in reading (i), a 31-frame median filter, and an
exponential rise fitted to frames 0-78 of the well with the lowest concentration above the EC50
(morse_figs3_rev cells 688-692), on the normalized dF/F0 and on the Eq. 8 concentration. The
EC50 and F are the claims file's CONSTANTS (Table 1 and the code's F), since the EC50s fitted here
do not reproduce Table 1; the fitted EC50s of reading (i) are a variant. tau above 200 frames is
dropped. Seconds = frames x 60/31. Fixing xinf to the median of frames 47-78, as the code does,
underestimates long time constants (a synthetic tau of 40 frames is fitted as 24.7); a variant fits
xinf with tau. Both labelling readings are shown.

In [ ]:
TAUS_BY = {}
for lname, L in READINGS:
    rows, taus = [], {}
    for g in M.GROUP_ORDER:
        if g[0] != g[1]:
            continue
        r = L == g[0]
        xb = M.normalized_dose_trace(DX[r], g)
        out = {"sensor": g[0], "cells": int(r.sum())}
        variants = [("const", M.KD_TABLE1[g], M.F_CODE[g], False), ("fitted", *FITS_BY[lname][("i", g)], False),
                    ("free xinf", M.KD_TABLE1[g], M.F_CODE[g], True)]
        for name, kd, f, free in variants:
            td, tc, w = M.taus_for_group(xb, g, kd, f, free_inf=free)
            if name == "const":
                taus[g] = (td, tc)
            out[f"{name} well"] = w
            out[f"{name} dF/F s"] = round(float(np.nanmedian(td)) * 60 / 31, 1) if len(td) else np.nan
            out[f"{name} conc s"] = round(float(np.nanmedian(tc)) * 60 / 31, 1) if len(tc) else np.nan
            out[f"{name} kept"] = f"{int(np.isfinite(td).sum())}/{int(np.isfinite(tc).sum())}"
        rows.append(out)
    te = pd.DataFrame(rows)
    TAUS_BY[lname] = taus
    if lname == "b":
        TE, TAUS = te, taus
    print(f"labels, reading ({lname}); variants: const (Table 1 EC50, the code's fit), fitted (EC50 of reading i), free xinf (the plateau fitted)")
    print(te.to_string(index=False))
    for name in ["const", "fitted", "free xinf"]:
        print(f"  {name}: sensors with median tau-on under 60 s, (a) dF/F0 {int((te[f'{name} dF/F s'] < 60).sum())} of 10, "
              f"(b) concentration {int((te[f'{name} conc s'] < 60).sum())} of 10")
    print()

**Result: reproduced, marginally, resting on the labelling, the Table 1 constants and the fixed
xinf.** With the reading
(b) labels and the code's fit, the median tau-on is under 60 s for 10 of 10 sensors on the
normalized dF/F0 (28.6 to 53.7 s) and for 8 of 10 on the concentration (34.4 to 58.9 s; AVP 74.2 s
and 5-HT 62.3 s are above). Both readings meet "at least 8 of 10". The result rests on the choices
around it: with the reading (a) labels the concentration fits give 7 of 10 (MT 63.6 s joins AVP
and 5-HT), so the two readings of the criterion would differ; with the EC50s fitted here in place
of Table 1's, 8 and 7 of 10 (reading b labels) and 10 and 8 (reading a); and with xinf fitted
instead of fixed, which lengthens the slow fits, 9 and 6 (b) and 8 and 6 (a). The concentration
count is exactly at the threshold (8 of 10), and each of these alternatives gives a split between
the criterion's two readings, so the outcome holds only under the main choices.

## figs3f-depth-delay

> Onset times increase only modestly with cell depth in the gel, with a positive slope for all
> sensors except ATP.

The concentration tau-on of figs3e against each cell's depth: the z centre of mass of its
reordered mask in the dose recording (the code uses the dose recording's masks, morse_figs3_rev
cell 707), relative to the probe's shallowest cell and converted at 9.89 um per plane. The code
plots depth as 9.89 x (15 - z) (cell 718), so depth from the front face grows as the plane index
falls; this orientation is taken from the code, and the opposite one is shown. A least-squares line
per sensor, pooled over probes. Both labelling readings are shown.

In [ ]:
zrel = np.concatenate([ZD[p] - ZD[p].min() for p in M.REV_PROBES])
depth = -M.UM_PER_PLANE * zrel
print("depth range (um, 0 = deepest-indexed plane):", round(depth.min(), 1), "to", round(depth.max(), 1))
for lname, L in READINGS:
    rows = []
    for g in M.GROUP_ORDER:
        if g[0] != g[1]:
            continue
        r = np.where(L == g[0])[0]
        tc = TAUS_BY[lname][g][1] * 60 / 31
        ok = np.isfinite(tc)
        sl = np.polyfit(depth[r][ok], tc[ok], 1)[0]
        rho = np.corrcoef(depth[r][ok], tc[ok])[0, 1]
        rows.append({"sensor": g[0], "cells": int(ok.sum()), "slope s/um (code orientation)": round(sl, 3),
                     "slope s/um (opposite)": round(-sl, 3), "r": round(rho, 2)})
    df = pd.DataFrame(rows)
    if lname == "b":
        DF = df
    pos = df["slope s/um (code orientation)"] > 0
    print(f"labels, reading ({lname})")
    print(df.to_string(index=False))
    print("  positive slopes, code orientation:", int(pos.sum()), "of 10; negative:", df.sensor[~pos].tolist())
    print("  positive slopes, opposite orientation:", int((~pos).sum()), "of 10")
    print("  largest |slope| (s/um):", df["slope s/um (code orientation)"].abs().max(), "\n")

**Result: not reproduced.** With the reading (b) labels, tau-on (concentration) rises with depth
for 5 of 10 sensors (AVP, 5-HT, NE, MT, HA) and falls for 5 (OXT, CCK, DA, ACh, ATP) in the
code's orientation of depth, and the opposite orientation gives the same split. With the reading
(a) labels, 7 of 10 rise (OXT, CCK and ATP fall), or 3 of 10 in the opposite orientation. No
variant reaches the 9 of 10 required. The slopes are modest (at most 0.52 s per um), and the
correlations weak (r 0.56 and 0.51 for MT, at most 0.24 in size for the other sensors).

## figs2f-cells-tracked

> About 70 to 130 active sensor cells per probe can be tracked across 2 to 8 recordings in
> different contexts.

A `matched<k>` file holds the masks matched between a reference recording and one other recording
of the probe (the k-th pair), and both files of a pair hold the same number of cells. The code
supports this reading: the mask matching (`5_caiman_fit_qc_matching/8_mask_matching.ipynb` cell 8)
writes the matched masks of one pair to both the reference and the moving recording, and
morse_figs2_2 cell 3 loads `..._matched{k}` for the k-th pair of a probe. The same matching notebook
(cells 15-17) loads the `matched1` file of 20230922-PL28 wells001 and saves it again as `matched`,
which may be why that probe's suffixes do not fit its pairs. The matched count of a pair is that
number. Files deposited twice are counted once, and the refit copies (`-refit-matched`, the same
masks) are left out. Per probe, the mean over its pairs.

The main result uses the probe sets of Figure S2F as listed in morse_figs2_2 cell 2: ten dipping
probes (40 pairs, the figure's n=40), 22 in vivo recordings, each matched to its post hoc dipping
(the figure says n=25), and 11 slice pairs (the figure says n=15), two of which are not in the
dandiset. All probes of the dandiset are shown as a variant (first cell).

In [ ]:
Mt = S[S.kind.str.match(r"^mr-raw-matched\d*$") & (S.subject != "0")].copy()
Mt["probe"] = Mt.subject.str.replace(r"-(wells\d*|slice|mouse\d?)$", "", regex=True)
Mt = Mt[Mt.subject != "250918-SL1ZS5-wells001"]
Mt["pair"] = Mt.kind.str.extract(r"matched(\d*)$")[0].replace("", "0").astype(int)
chk = Mt.groupby(["probe", "pair"]).agg(n_cells=("cells", "nunique"), counts=("cells", lambda c: sorted(set(c))),
                                        files=("subject", lambda c: sorted(c)))
print("probe and pair suffixes whose files disagree on the count:")
print(chk[chk.n_cells > 1].to_string())
# The reference recording of a probe holds one matched<k> file per pair; take the pairs from it.
nref = Mt.groupby(["probe", "subject"]).size().rename("n").reset_index()
ref = nref.sort_values(["probe", "n", "subject"], ascending=[True, False, True]).groupby("probe").head(1)
pairs = Mt.merge(ref[["probe", "subject"]], on=["probe", "subject"]).groupby(["probe", "pair"]).agg(
    cells=("cells", "first"), contexts=("context", "first"))
ctx = Mt.groupby("probe").context.agg(lambda c: "+".join(sorted(set(c))))
pairs = pairs.reset_index().merge(ctx.rename("ctx").reset_index(), on="probe").set_index(["probe", "pair"])
pairs["contexts"] = pairs["ctx"]
print("\npairs (from each probe's reference recording):", len(pairs))
pp = pairs.reset_index().groupby("probe").agg(pairs=("pair", "count"), mean_matched=("cells", "mean"),
                                                 min_matched=("cells", "min"), max_matched=("cells", "max"),
                                                 contexts=("contexts", lambda c: ",".join(sorted(set(c)))))
pp["kind"] = np.where(pp.contexts.str.contains("mouse"), "in vivo", np.where(pp.contexts.str.contains("slice"), "slice", "dipping"))
inr = (pp.mean_matched >= 63) & (pp.mean_matched <= 143)
print(f"\nprobes: {len(pp)}; per-probe mean matched cells: median {pp.mean_matched.median():.0f}, "
      f"range {pp.mean_matched.min():.0f} to {pp.mean_matched.max():.0f}")
print(f"probes whose mean lies in 63-143: {int(inr.sum())} of {len(pp)} ({inr.mean():.2f})")
print(pp.assign(in_range=inr).groupby("kind").agg(probes=("pairs", "count"), pairs=("pairs", "sum"),
      median=("mean_matched", "median"), lo=("mean_matched", "min"), hi=("mean_matched", "max"), in_range=("in_range", "sum")).to_string())
print("\nrecordings per probe (pairs + 1): range", pp.pairs.min() + 1, "to", pp.pairs.max() + 1)
code_set = [p for p in pp.index if p.startswith("202309")]
cs = pp.loc[code_set]
print(f"\nthe code's dipping set (morse_figs2_2 cell 2, probes of 2023-09-15 to 2023-09-27): {len(cs)} probes, {cs.pairs.sum()} pairs; "
      f"per-probe means {np.round(cs.mean_matched.values, 1).tolist()}; in 63-143: {int(((cs.mean_matched >= 63) & (cs.mean_matched <= 143)).sum())} of {len(cs)}")

In [ ]:
# The probe sets of Figure S2F as the code lists them (morse_figs2_2 cell 2): dipping, ten probes whose
# reference wells001 recording is matched to each later recording (pair k in the reference's matched<k>);
# in vivo, each mouse recording matched to its post hoc dipping (pair 0); slice, each slice recording matched
# to its dipping (pair 0, or 1 where the code says so).
code_dip = {"20230915-PL52": 4, "20230916-PL48": 1, "20230916-PL54": 5, "20230920-PL58": 3, "20230922-PL28": 7,
            "20230923-PL35": 5, "20230925-PL58": 4, "20230926-PL56": 5, "20230927-PL34": 3, "20230927-PL58": 3}
code_vivo = ["220216-PK25PL8", "220311-PK24PL9", "220406-PK24PL10", "220408-PK27PL20", "220427-PK27PL20", "220622-PK44PL17",
             "220624-PK45PL13", "220701-PK49PL18", "220903-PK63PL34", "221018-PS10PL24", "230519-DG41PL52", "230525-DG42PL39",
             "230526-PK84PL50", "230726-DG47PL54", "230802-DG48PL58", "231018-DG52PL66", "231024-DG54PL61", "231025-DG56PL62",
             "231030-DG60PL67", "231105-DG39PL32", "231107-DG44PL58", "231108-DG61PL62"]
code_slice = [("20230509-DA1PL53", 0), ("20230607-SL1PL47", 0), ("20230801-SL1PL35", 0), ("20230817-SL1PL55", 0),
              ("20230817-SL1PL55", 1), ("20230822-SL2PL52", 0), ("20230824-SL1PL60", 0), ("20230824-SL1PL60", 1),
              ("20230829-SL1PL56", 0), ("20230830-SL1PL28", 0), ("20230831-SL1PL55", 0)]
cells_of = {(r.subject, r.pair): r.cells for r in Mt.itertuples()}
rows = []
for p, n in code_dip.items():
    for k in range(n):
        rows.append({"set": "dipping", "probe": p, "pair": k, "cells": cells_of.get((p + "-wells001", k))})
for p in code_vivo:
    rows.append({"set": "in vivo", "probe": p, "pair": 0, "cells": cells_of.get((p + "-mouse", 0))})
for p, k in code_slice:
    rows.append({"set": "slice", "probe": p, "pair": k, "cells": cells_of.get((p + "-slice", k))})
CS = pd.DataFrame(rows)
print("pairs of the code's sets:", CS.groupby("set").size().to_dict(), "; missing from the dandiset:",
      CS[CS.cells.isna()].groupby("set").size().to_dict(), CS[CS.cells.isna()].probe.unique().tolist())
# Check that both files of a pair, the reference recording's matched<k> and the moving recording's
# matched<k>, hold the same count. A pair with one file in the dandiset cannot be checked.
nf = Mt.groupby(["probe", "pair"]).agg(files=("subject", "nunique"), counts=("cells", "nunique")).reset_index()
nf = CS[["set", "probe", "pair"]].merge(nf, on=["probe", "pair"])
nf["two_files"] = nf.files >= 2
nf["agree"] = nf.two_files & (nf.counts == 1)
print("pairs of the code's sets with matched files, by the number of files with their suffix:")
print(nf.groupby("set").agg(pairs=("files", "size"), two_files=("two_files", "sum"), counts_agree=("agree", "sum")).to_string())
print("pairs with two files whose counts differ:", nf[nf.two_files & ~nf.agree][["probe", "pair"]].values.tolist())
print("pairs with one file only:", nf[~nf.two_files][["probe", "pair"]].values.tolist())
print("dipping pairs k >= 1 with two files:", int((nf.two_files & (nf.set == "dipping") & (nf.pair >= 1)).sum()),
      "; of them with equal counts:", int((nf.agree & (nf.set == "dipping") & (nf.pair >= 1)).sum()))
CSp = CS.dropna().groupby(["set", "probe"]).cells.mean().reset_index()
CSp["in 63-143"] = (CSp.cells >= 63) & (CSp.cells <= 143)
print(CSp.groupby("set").agg(probes=("probe", "count"), median=("cells", "median"), lo=("cells", "min"), hi=("cells", "max"),
                             in_range=("in 63-143", "sum")).to_string())
print(f"the code's sets pooled: {int(CSp['in 63-143'].sum())} of {len(CSp)} probe means in 63-143 ({CSp['in 63-143'].mean():.2f}); "
      f"pair counts range {int(CS.cells.min())} to {int(CS.cells.max())}")

**Result: reproduced.** In the code's Figure S2F sets, 26 of 39 probes (0.67) have a mean matched
count within 63 to 143: 8 of 10 dipping probes (means 56.5 to 108.0), 15 of 22 in vivo probes (34
to 103) and 3 of the 7 slice probes present in the dandiset (27 to 101); single pairs hold 27 to
108 cells. Of the 71 pairs of these sets with a matched file in the dandiset, 70 have both files,
the reference recording's and the moving recording's `matched<k>` (20230922-PL28 pair 3 has only
the reference's), and in 69 of the 70 the two hold the same count (20230922-PL28 pair 0 does not).
The 70 include 29 dipping pairs with k of 1 or more, all of which agree. Over all 72 probes of the dandiset and 110 pairs, 44 (0.61) are in range (15 of 21
dipping, 24 of 40 in vivo, 5 of 11 slice), with 2 to 8 recordings per probe. More than half are in
range in both sets, though the slice probes alone are not.

## Blocked claims: what the files hold

The cell below shows, by context, what the files hold of stimulus, acquisition, region or sensor
information, which the slice and in vivo claims need, and lists the mouse probes that have a post hoc
dipping recording (asked by the verifier for fig3g-ten-sensor-yield). Besides the acquisition and
stimulus groups, it lists the descriptive fields where a region, an experiment or a sensor label
could be stored: the imaging plane's location and description, the session description, the
columns of the mask table, the subject fields and the analysis group. In every context these hold
the same placeholders (location "brain") and no label column or analysis object.

In [ ]:
# What the blocked claims need and what the files hold, by context (the survey above).
ctx = S[S.subject != "0"].groupby("context").agg(subjects=("subject", "nunique"), files=("path", "count"),
                                                 acquisition=("acquisition", "sum"), stimulus=("stimulus", "sum"),
                                                 rates=("rate_hz", lambda v: sorted(set(v))), starts=("start", "nunique"))
print(ctx.to_string())
# The descriptive fields where a region, an experiment or a sensor label could be stored: the imaging plane's
# location and description, the session description, the columns of the mask table, the subject fields, and
# the number of analysis objects.
for c in ["location", "plane_description", "session_description", "roi_columns", "subject_fields", "analysis"]:
    print(f"\n{c}, distinct values by context:", S[S.subject != "0"].groupby("context")[c].agg(lambda v: sorted(set(v))).to_dict())
# Mouse subjects with a post hoc dipping recording, and their length in 3-minute wells at 31 (or 52) frames per minute.
names = S[S.subject != "0"].subject.unique()
mice = sorted({s.rsplit("-", 1)[0] for s in names if s.endswith(("-mouse", "-mouse2"))})
rows = []
for m in mice:
    dips = sorted(s for s in names if s.startswith(m + "-wells"))
    for d in dips:
        fr = int(S[S.subject == d].frames.max())
        rows.append({"probe": m, "dipping": d.split("-")[-1], "frames": fr, "wells at 31/min": round(fr / 93, 2), "wells at 52/min": round(fr / 156, 2)})
dipv = pd.DataFrame(rows)
print(f"\nmouse probes: {len(mice)}; with a post hoc dipping recording: {dipv.probe.nunique()}")
print(dipv.to_string(index=False))

### fig3g-ten-sensor-yield, in part

> With ten-sensor probes in CSF of four mice, up to 9 sensors per probe were recovered (median 8.5),
> with on average 3.8 cells per sensor per probe.

The claim stays blocked: two of its four probes cannot be labelled. The two 2025 probes are labelled
here as the revision probes are, as a partial check.

In [ ]:
# Part of fig3g on the two 2025 ten-sensor in vivo probes. Their post hoc dipping (wells101) has the revision
# probes' 11-well order (morse_fig3_rev cells 51-52, 56); the two 2023 probes of Figure 3G are recorded at 52
# frames per minute, and their labels are loaded from files not in the code repository (cell 170), so they
# cannot be labelled here.
t0 = time.time()
V = [M.load_traces(M.file_path(p, "wells101")) for p in ["ZS93ZS22-251121", "ZS94ZS19-251122"]]
print("frames:", [v.shape[1] for v in V], "cells:", [v.shape[0] for v in V], f"loaded in {time.time() - t0:.0f} s")
pv = [M.id_preprocess(v) for v in V]
va, vb, vg, _ = M.identify([x[1] for x in pv])
for name, lab in [("(a) pooled", va), ("(b) pooled + per probe", vb)]:
    y = M.yield_stats(lab, vg, n_probes=2)
    print(f"{name}: labelled cells {y['cells']}; sensors per probe {y['sensors_per_probe']}; "
          f"cells per sensor per probe {y['cells_per_sensor_per_probe']:.2f}")
print("the code prints 9, 8, 9 and 5 sensors for its four probes (morse_fig3_rev cell 204), the two 2025 probes presumably the last two")

**Result: blocked (metadata insufficient).** Of the four probes of Figure 3G, the two 2023 ones
(231105-DG39PL32, 231107-DG44PL58) have a post hoc dipping of 3588 frames, 23 wells at the 52
frames per minute the code uses for them, whose well order is in neither the files nor the code,
and the code loads their cluster labels from files outside its repository (morse_fig3_rev cell
170). The two 2025 probes have 1023 frames, the revision probes' 11 wells; labelled as those are,
they carry 7 and 6 sensors (reading b; 6 and 6 in reading a) and 7.15 cells per sensor per probe
(6.83). The code prints 9, 8, 9 and 5 sensors for its four probes. The median and the mean over
four probes cannot be computed from two.

## Summary

| Claim | Outcome |
|---|---|
| link-revision-well-order | reproduced |
| fig1c-cells-per-gel | reproduced (reading b; assumption on the gel thickness) |
| fig1g-sensor-yield | partially reproduced (assumed labelling) |
| fig1g-all-ten-sensors-respond | not reproduced as written (assumed labelling); every sensor's largest rise is in its own well; asked of a reader |
| fig1h-crosstalk-pairs | not reproduced (assumed labelling) |
| table1-ec50-on-target | not reproduced (assumed labelling and normalization) |
| table1-ec50-off-target | not reproduced (assumed labelling and normalization) |
| table1-peak-dff-morse | not reproduced (assumed labelling) |
| fig1j-ec50-consistent-across-probes | not reproduced (assumed labelling and normalization) |
| figs3e-tau-on-under-one-minute | reproduced, marginal (assumed labelling; Table 1 constants; fixed xinf; each alternative gives a split) |
| figs3f-depth-delay | not reproduced (assumed labelling; depth orientation from the code) |
| figs2f-cells-tracked | reproduced (the code's Figure S2F sets; assumed reading of the matched files) |
| fig1d, fig1f, methods-revision-probe-stability, figs2c, slices-probe-stability | blocked: data missing |
| figs6ef, fig4-ex-vivo-elevations, figs6j, fig5gh, fig6, fig6c, figs6a | blocked: data missing |
| fig4-ex-vivo-two-orders-lower | blocked: data missing (depends on fig4-ex-vivo-elevations) |
| the 16 slice claims, fig3df, fig3g (two of four probes labelled, shown), fig3h, fig4-ip-csf-concentrations, fig5ab, fig5cd | blocked: metadata insufficient |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The sensor labels are made by the Methods'
clustering without the code's hand corrections, so the labelled cells are not the paper's 506.
Matching clusters to ligands uses two assumptions (a boxcar over the first minute of each well, and
the peak rule); both are shown changed, and every labelled claim is shown under both labelling
readings. Of the outcomes, only figs3e-tau-on-under-one-minute depends on the labelling reading. The
dose-response normalization follows the code's baseline and peak wells, since the Methods do not
fix it, without the code's per-cell background fits over hand-set windows; the code's noise cut is
shown as a variant. The time base is the code's 31 frames per minute, since the files' 30 Hz is a
placeholder. tau-on uses the Table 1 EC50s to pick the well and convert to concentration.

What was found about the dataset. The files hold traces and masks only: no sensor labels, no
well, wash-on, injection or stimulation times, and no region or experiment labels, so the slice and
in vivo claims cannot be tested. The timing metadata are placeholders (30 Hz, every file starting
at 0 on 2023-01-01), and the indicator is given as OGB-1. The 3D masks are stored scrambled
(Fortran-order data in a C-order array). 67 files of an unnamed subject `0` carry no context; 16
of them repeat named files. The slice and wells001 files of 250918-SL1ZS5 are identical. In
20230922-PL28 the matched-file suffixes do not match their pairs (wells201 holds `matched` and
`matched1` as copies, wells401's `matched` file has the count of wells001's `matched3`), and the
251119-ZS16 pair has 143 against 142 cells.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")